# RAG för klassificering av SNI 2025

Textklassificering genom att bygga ett enkelt system för retrieval‑augmented generation (RAG). Vi kommer att:
- bygga en kunskapsbas med hjälp av SNI2025 omfattartexter (samma uppgifter som finns i https://snisok.scb.se/)
- hämta de "top_k" mest relevanta koderna (semantisk likhet) baserat på en mindre språkmodell.
- använda en stor språkmodell (LLM) för att göra den slutliga klassificeringen.

Installera följande python-paket i terminalen:
pip install torch sentence-transformers transformers openpyxl openai

In [ ]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from transformers import pipeline

## Del 1 - Bygga en kunskapsbas

Det första steget i varje RAG‑system är att bygga en kunskapsbas.
I sammanhanget retrieval‑augmented generation är en kunskapsbas en strukturerad samling av extern information (t.ex. dokument, beskrivningar osv.) som kan genomsökas och hämtas vid behov. Syftet är vanligtvis att förankra språkmodellens svar i faktabaserad information i stället för att enbart förlita sig på modellens interna kunskap, eller att komplettera modellen med kunskap som inte fanns tillgänglig i träningsdatan.
För att bygga en kunskapsbas behöver vi:
1) Definiera vilka dokument som ska lagras och hur de ska struktureras.
2) Skapa inbäddningar (embeddings) av de utvalda texterna med hjälp av SentenceTransformer.
3) Lagra embeddingarna i minnet eller i en vektordatabas.

### Definiera och strukturera dokument
I vårt fall kommer kunskapsbasen att bestå av SNI-koder på  2‑siffrig nivå. Därför är det första steget att ladda in filen som innehåller samtliga SNI-koder:

In [ ]:
df_sni2025 = pd.read_excel("https://minio.lab.sspcloud.fr/scbmrid/OmfattartexterSNI2025.xlsx")

df_sni2025.head()

Sedan plockar vi ut informationen på 2-siffrig nivå:

In [ ]:
df_sni2025_two = df_sni2025[df_sni2025["Niva"] == 2].reset_index(drop=True)

df_sni2025_two

Som vi kan se innehåller datamängden flera textkolumner, såsom rubriker och beskrivningar (t.ex. Omfattar, Omfattar även, osv.).

Generellt gäller att ju mer sammanhang vi ger till embedding‑modellen, desto bättre blir resultatet, förutsatt att den slutliga texten som ska bäddas in inte blir för lång.
Ett enkelt sätt att utnyttja den textuella information som finns i SNI‑datamängden är att kombinera olika textfält i en strukturerad mall. Vi väljer att för varje SNI-kod plocka ut "Rubrik" och informationen "Omfattar".

In [ ]:
codes = df_sni2025_two["SNI2025"].tolist()
titles = df_sni2025_two["Rubrik"].tolist()
descriptions = df_sni2025_two["Omfattar"].tolist()

descriptor_template = "{title}.\n{description}"

descriptors = []
for title, description in zip(titles, descriptions):
    descriptors.append(
        descriptor_template.format(title=title.upper(), description=description)
    )

print(descriptors[5])

### Skapa inbäddningar (embeddings) för descriptors
Nu när vi har byggt en deskriptor för var och en av de 87 SNI-koderna behöver vi skapa inbäddningar av dem med hjälp av en meningsbaserad embedding‑modell. Det finns många sådana modeller tillgängliga på Hugging Face. Vi kan ladda dem med hjälp av klassen SentenceTransformer från biblioteket sentence_transformers.
I detta exempel använder vi Kungliga bibliotekets modell "KBLab/sentence-bert-swedish-cased".

In [ ]:
model = SentenceTransformer("KBLab/sentence-bert-swedish-cased")

Efter att modellen har laddats kan vi skapa inbäddningar för deskriptorerna. Varje deskriptor kommer då att representeras som en inbäddningsvektor med 768 dimensioner.

In [9]:
embeddings = model.encode(descriptors, convert_to_tensor=True)

Eftersom vi endast har 87 SNI-koder (och därmed 87 vektorer) kan vi i detta fall lagra embeddingarna direkt i minnet utan att behöva använda en vektordatabas.

## Del 2 - Söka i kunskapsbasen
Denna fas består i att utveckla en metod för att söka i kunskapsbasen med hjälp av semantisk likhet.
Vid sökning i en kunskapsbas bäddas en fråga i naturligt språk in med samma modell som användes för att bygga kunskapsbasen. Därefter beräknas ett avståndsmått för att utvärdera likheten mellan frågan och dokumenten i kunskapsbasen.
Det vanligaste avståndsmåttet som används för att hämta element från en kunskapsbas är cosinuslikhet (https://en.wikipedia.org/wiki/Cosine_similarity)
Nu kan vi slutligen definiera en funktion som, givet en fråga,  hitta de top_k dokumenten i kunskapsbasen som är mest lika frågan, baserat på cosinuslikhet mellan embeddings:

In [10]:
def search_base(
    query: str,
    base: torch.Tensor,
    embedding_model: SentenceTransformer,
    top_k: int = 5
):
    query_embedding = embedding_model.encode(query, convert_to_tensor=True)
    cos_scores = torch.nn.functional.cosine_similarity(base, query_embedding)
    top_results = torch.topk(cos_scores, k=top_k)

    return top_results

Låt oss testa funktionen med en exempel­fråga. Som resultat får vi både värden (likhetsmått) och index. För att identifiera de motsvarande SNI-koderna kan vi använda indexvärdena för att filtrera SNI‑dataframen.

In [ ]:
sample_query = "Verksamhet som avser utvinning av mineraler"


top_results = search_base(sample_query, embeddings, model, top_k=5)

for i, sim in zip(top_results.indices, top_results.values):
    code = df_sni2025_two.iloc[int(i)]["SNI2025"]
    title = df_sni2025_two.iloc[int(i)]["Rubrik"]
    print(f"{code}: {title}\nSimilarity: {sim:.3f}\n")

Vi skulle kunna stanna här och helt enkelt välja det första resultatet som vår klassificeringsgissning. En mer robust(?) metod är dock att använda en LLM som domare för att fatta det slutliga beslutet mellan kandidaterna.

## Del 3 - Slutlig klassificering med LLM
Detta sista steg innefattar en generativ stor språkmodell (LLM) som får i uppgift att välja ett alternativ bland de framtagna kandidatkoderna. Först laddar vi en LLM‑modell som finns tillgänlig i SSP Cloud.

In [12]:
from openai import OpenAI
import os

# Hämta API-nyckeln från miljövariabler (injiceras via My Secrets)
api_key = api_key = os.environ["SSP_API_KEY"]

# Välj modell manuellt (i Open WebUI kan du se tillgängliga modeller)
model_LLM = "gpt-oss:20b"

# Initiera klienten med SSP Clouds LLM-endpoint
client = OpenAI(api_key=api_key,
base_url="https://llm.lab.sspcloud.fr/api")


### Systemprompt och instruktioner
Vi behöver definiera en systemprompt, dvs. riktlinjer som hjälper LLM:n att följa våra instruktioner, samt den faktiska prompt‑mallen som används för att strukturera indata till modellen.

In [13]:
SYSTEM_PROMPT: str = """Du är en klassificerare av ekonomiska aktiviteter.

Baserat på en beskrivning ska du ENDAST returnera den mest lämpliga kandidaten, inklusive både kod och titel."""

PROMPT_TEMPLATE: str = """BESKRIVNING: "{query}"

KANDIDATER:\n{candidates}"""

### Strukturering av indata
Vi behöver extrahera listan med kandidater genom semantisk likhetsbaserad sökning och strukturera den på ett sätt som gör den begriplig för LLM:n.

In [14]:
sample_query = "Verksamhet som avser utvinning av mineraler"

top_results = search_base(sample_query, embeddings, model, top_k=5)

candidates_list = []
for i, sim in zip(top_results.indices, top_results.values):
    code = df_sni2025_two.iloc[int(i)]["SNI2025"]
    title = df_sni2025_two.iloc[int(i)]["Rubrik"]
    candidates_list.append(f"{code} - {title}")

candidates = "\n".join(candidates_list)

In [ ]:
parsed_prompt = PROMPT_TEMPLATE.format(query=sample_query, candidates=candidates)

print(parsed_prompt)

### Fullständigt RAG‑system
Nu kan vi testa det fullständiga RAG‑systemet för att klassificera den ekonomiska aktiviteten baserat på en fråga i naturligt språk.

In [ ]:
# Skicka ett enkelt testmeddelande
response = client.chat.completions.create(
    model=model_LLM,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": parsed_prompt}
    ])

# Skriv ut svaret
print(response.choices[0].message.content)

## En interaktiv demo
Slutligen kan vi skapa en liten interaktiv demo för vårt RAG‑system. Först behöver vi definiera en rag‑funktion som utför hela pipelinen:

In [56]:
def rag(
    query: str,
    embeddings: torch.Tensor,
    embedding_model: SentenceTransformer,
    openai_client: OpenAI,
    llm_model: str,
    system_prompt: str,
    prompt_template: str,
    nace_df: pd.DataFrame,
    top_k: int = 5
):
    # 1. Semantic search
    top_results = search_base(
        query=query,
        base=embeddings,
        embedding_model=embedding_model,
        top_k=top_k
    )

    # 2. Build candidate list
    candidates_list = []
    for i, sim in zip(top_results.indices, top_results.values):
        code = nace_df.iloc[int(i)]["SNI2025"]
        title = nace_df.iloc[int(i)]["Rubrik"]
        candidates_list.append(f"{code} - {title}")

    candidates = "\n".join(candidates_list)

    # 3. Prompt
    parsed_prompt = prompt_template.format(
        query=query,
        candidates=candidates
    )

    # 4. LLM call
    response = openai_client.chat.completions.create(
        model=llm_model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": parsed_prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip()

Sedan kan vi köra demon:

In [ ]:
print("SNI 2025 - RAG demo")
print("Skriv EXIT för att avsluta\n")

query = input("Ange verksamhetsbeskrivning: ")

while query != "EXIT":
    result = rag(
        query=query,
        embeddings=embeddings,
        embedding_model=model,
        openai_client=client,
        llm_model=model_LLM,
        system_prompt=SYSTEM_PROMPT,
        prompt_template=PROMPT_TEMPLATE,
        nace_df=df_sni2025_two,
        top_k=5
    )

    print(f"\nRESULTAT: {result}\n")
    query = input("Ange verksamhetsbeskrivning: ")